# 02. Classical Models

This notebook is a **small, presentation-friendly classical forecasting study** built only on the **4 representative series** selected in `01_EDA.ipynb`:

- longest history
- highest total weight
- most volatile
- most stable

Instead of evaluating hundreds of panel series, this notebook asks a simpler question: **how do a few non-trivial classical models behave on four very different time series from the dataset?**

**What this notebook does**

1. recreate the same 4 series chosen in the EDA
2. show their raw behavior and quick stationarity diagnostics
3. run a compact lineup of rolling-average, smoothing, and small ARIMA models
4. compare models with tables, heatmaps, fit-status charts, and forecast overlays
5. summarize what classical methods seem to work best for each type of series


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=ConvergenceWarning)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data' / 'ts-forecasting'
VAL_CUTOFF = 2880
SEED = 42

train = pd.read_parquet(
    DATA_DIR / 'train.parquet',
    columns=['code', 'sub_code', 'sub_category', 'horizon', 'ts_index', 'y_target', 'weight'],
)
series_keys = ['code', 'sub_code', 'sub_category', 'horizon']
train = train.sort_values(series_keys + ['ts_index']).reset_index(drop=True)
series_grouped = train.groupby(series_keys, sort=False)

REASON_ORDER = ['longest history', 'highest total weight', 'most volatile', 'most stable']

print(f'Train rows: {len(train):,}')
print(f'VAL_CUTOFF: {VAL_CUTOFF}')
print(f'Series keys: {series_keys}')
    

## 1. Recreate the 4 representative series from `01_EDA.ipynb`

The EDA notebook picked these four using the following rule:

- only consider series that **cross the validation cutoff**
- require `length >= 120`
- then choose one series each for:
  - longest history
  - highest total weight
  - most volatile (`target_std` largest)
  - most stable (`target_std` smallest but still positive)

This section recreates that exact logic and checks that the selected series match the EDA output.
    

In [ ]:
series_stats = (
    train.groupby(series_keys)
         .agg(
             length=('ts_index', 'size'),
             start=('ts_index', 'min'),
             end=('ts_index', 'max'),
             total_weight=('weight', 'sum'),
             target_std=('y_target', 'std'),
         )
         .reset_index()
)
series_stats['crosses_cutoff'] = (series_stats['start'] <= VAL_CUTOFF) & (series_stats['end'] > VAL_CUTOFF)

eligible = series_stats[(series_stats['crosses_cutoff']) & (series_stats['length'] >= 120)].copy()
eligible['target_std'] = eligible['target_std'].fillna(0.0)
stable_pool = eligible[eligible['target_std'] > 0].copy()

chosen = pd.concat([
    eligible.nlargest(1, 'length').assign(reason='longest history'),
    eligible.nlargest(1, 'total_weight').assign(reason='highest total weight'),
    eligible.nlargest(1, 'target_std').assign(reason='most volatile'),
    stable_pool.nsmallest(1, 'target_std').assign(reason='most stable'),
], ignore_index=True).drop_duplicates(subset=series_keys)

chosen['reason'] = pd.Categorical(chosen['reason'], categories=REASON_ORDER, ordered=True)
chosen = chosen.sort_values('reason').reset_index(drop=True)

expected = pd.DataFrame([
    {'code': 'X9BZ68VQ', 'sub_code': 'OYJGNSQK', 'sub_category': 'DPPUO5X2', 'horizon': 1,  'reason': 'longest history'},
    {'code': 'SJZP0OVU', 'sub_code': 'OYJGNSQK', 'sub_category': 'NQ58FVQM', 'horizon': 25, 'reason': 'highest total weight'},
    {'code': 'W4S29LF4', 'sub_code': 'KL66VIS3', 'sub_category': 'PHHHVYZI', 'horizon': 25, 'reason': 'most volatile'},
    {'code': 'SJZP0OVU', 'sub_code': 'OYJGNSQK', 'sub_category': 'NQ58FVQM', 'horizon': 1,  'reason': 'most stable'},
])
expected['reason'] = pd.Categorical(expected['reason'], categories=REASON_ORDER, ordered=True)
expected = expected.sort_values('reason').reset_index(drop=True)

actual_keys = chosen[[*series_keys, 'reason']].reset_index(drop=True)
pd.testing.assert_frame_equal(actual_keys, expected, check_dtype=False)

chosen['series_label'] = chosen['reason'].astype(str).str.title()
display(chosen[['series_label', *series_keys, 'length', 'total_weight', 'target_std']].round({'total_weight': 2, 'target_std': 4}))
    

## 2. Raw series overview

Before fitting models, it helps to look at the four selected series directly. The four picks are intentionally very different, so we should expect different models to behave differently on them as well.
    

In [ ]:
def get_series(row):
    key = (row['code'], row['sub_code'], row['sub_category'], row['horizon'])
    return series_grouped.get_group(key)[['ts_index', 'y_target', 'weight']].reset_index(drop=True)


def split_series(series_df):
    train_part = series_df[series_df['ts_index'] <= VAL_CUTOFF].copy()
    val_part = series_df[series_df['ts_index'] > VAL_CUTOFF].copy()
    return train_part, val_part

series_summary_rows = []
for _, row in chosen.iterrows():
    s = get_series(row)
    train_part, val_part = split_series(s)
    series_summary_rows.append({
        'series_label': row['series_label'],
        'train_len': len(train_part),
        'val_len': len(val_part),
        'mean_y': s['y_target'].mean(),
        'std_y': s['y_target'].std(),
        'total_weight': s['weight'].sum(),
    })
series_summary = pd.DataFrame(series_summary_rows)
display(series_summary.round({'mean_y': 4, 'std_y': 4, 'total_weight': 2}))

fig, axes = plt.subplots(len(chosen), 1, figsize=(13, 10), sharex=False)
axes = np.atleast_1d(axes)
for ax, (_, row) in zip(axes, chosen.iterrows()):
    s = get_series(row)
    ax.plot(s['ts_index'], s['y_target'], linewidth=1.1, color='#4C72B0')
    ax.axvline(VAL_CUTOFF, color='black', linestyle='--', linewidth=1)
    ax.set_title(f"{row['series_label']} — {row['code']} / {row['sub_code']} / H{row['horizon']}")
    ax.set_ylabel('y_target')
axes[-1].set_xlabel('ts_index')
plt.tight_layout()
plt.show()
    

## 3. Quick diagnostics on the 4 series

This is intentionally lightweight. We only check whether the 4 series look more stationary after first differencing, because that helps interpret why some ARIMA specifications may work better than others.
    

In [ ]:
def safe_adf(series):
    y = pd.Series(series).dropna().to_numpy(dtype=float)
    if len(y) < 20 or np.nanstd(y) == 0:
        return np.nan
    try:
        return float(adfuller(y, autolag='AIC')[1])
    except Exception:
        return np.nan


diagnostic_rows = []
for _, row in chosen.iterrows():
    s = get_series(row)
    y = s['y_target'].reset_index(drop=True)
    train_part, val_part = split_series(s)
    diagnostic_rows.append({
        'series_label': row['series_label'],
        'level_adf_p': safe_adf(y),
        'diff_adf_p': safe_adf(y.diff().dropna()),
        'train_len': len(train_part),
        'val_len': len(val_part),
    })
diagnostic_df = pd.DataFrame(diagnostic_rows)
display(diagnostic_df.round(4))

fig, axes = plt.subplots(len(chosen), 1, figsize=(13, 9), sharex=False)
axes = np.atleast_1d(axes)
for ax, (_, row) in zip(axes, chosen.iterrows()):
    s = get_series(row)
    y_diff = s['y_target'].diff()
    ax.plot(s['ts_index'], y_diff, linewidth=1.0, color='#55A868')
    ax.axvline(VAL_CUTOFF, color='black', linestyle='--', linewidth=1)
    ax.set_title(f"First difference — {row['series_label']}")
    ax.set_ylabel('diff(y)')
axes[-1].set_xlabel('ts_index')
plt.tight_layout()
plt.show()
    

## 4. Classical model lineup

To keep this notebook simple, we use a **small, non-trivial lineup**:

- `Rolling Mean (w=10)`
- `SES`
- `Holt`
- `ARIMA(1,0,0)`
- `ARIMA(0,1,1)`
- `ARIMA(1,1,0)`
- `ARIMA(1,1,1)`

The earlier version also included `Zero`, `Naive`, and `Drift`, but those dominate the plots with flat or straight-line forecasts and do not add much insight for this 4-series comparison. The remaining set still gives us one simple average baseline, two smoothing models, and four small ARIMA variants.


In [ ]:
def weighted_skill(y_true, y_pred, weight):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    weight = np.asarray(weight, dtype=float)
    denom = np.sum(weight * (y_true ** 2))
    if denom == 0:
        return np.nan
    ratio = np.sum(weight * ((y_true - y_pred) ** 2)) / denom
    ratio = min(max(ratio, 0.0), 1.0)
    return float(np.sqrt(1.0 - ratio))


def weighted_rmse(y_true, y_pred, weight):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    weight = np.asarray(weight, dtype=float)
    if np.sum(weight) == 0:
        return np.nan
    return float(np.sqrt(np.sum(weight * ((y_true - y_pred) ** 2)) / np.sum(weight)))


def weighted_mae(y_true, y_pred, weight):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    weight = np.asarray(weight, dtype=float)
    if np.sum(weight) == 0:
        return np.nan
    return float(np.sum(weight * np.abs(y_true - y_pred)) / np.sum(weight))


def mase(y_true, y_pred, train_scale):
    if not np.isfinite(train_scale) or train_scale == 0:
        return np.nan
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)) / train_scale)


def forecast_rolling_mean(y, steps, window=10):
    w = min(window, len(y))
    return np.repeat(float(y.iloc[-w:].mean()), steps)


def forecast_ses(y, steps):
    fit = SimpleExpSmoothing(y, initialization_method='estimated').fit(optimized=True)
    return np.asarray(fit.forecast(steps), dtype=float)


def forecast_holt(y, steps):
    fit = Holt(y, initialization_method='estimated').fit(optimized=True)
    return np.asarray(fit.forecast(steps), dtype=float)


def make_arima_forecaster(order):
    def _forecast(y, steps):
        fit = ARIMA(
            y,
            order=order,
            enforce_stationarity=False,
            enforce_invertibility=False,
        ).fit()
        pred = np.asarray(fit.forecast(steps), dtype=float)
        if not np.all(np.isfinite(pred)):
            raise ValueError('non-finite forecast')
        return pred
    return _forecast


MODEL_SPECS = {
    'Rolling Mean (w=10)': {'min_train': 2, 'func': lambda y, s: forecast_rolling_mean(y, s, 10)},
    'SES': {'min_train': 3, 'func': forecast_ses},
    'Holt': {'min_train': 5, 'func': forecast_holt},
    'ARIMA(1,0,0)': {'min_train': 20, 'func': make_arima_forecaster((1, 0, 0))},
    'ARIMA(0,1,1)': {'min_train': 20, 'func': make_arima_forecaster((0, 1, 1))},
    'ARIMA(1,1,0)': {'min_train': 20, 'func': make_arima_forecaster((1, 1, 0))},
    'ARIMA(1,1,1)': {'min_train': 20, 'func': make_arima_forecaster((1, 1, 1))},
}

model_overview = pd.DataFrame(
    {
        'model': list(MODEL_SPECS),
        'min_train': [spec['min_train'] for spec in MODEL_SPECS.values()],
    }
)
display(model_overview)


## 5. Evaluate the models on the 4 representative series

Each model is fit on the pre-cutoff train segment and forecast over the full post-cutoff validation segment. We keep the logic simple and record:

- `skill_score`
- weighted RMSE
- weighted MAE
- MASE
- model `status`

A few ARIMA fits are expected to be skipped because two of the representative series only have 7 pre-cutoff training points.


In [ ]:
results_rows = []
predictions = {}

for _, row in chosen.iterrows():
    series_df = get_series(row)
    train_part, val_part = split_series(series_df)
    y_train = train_part['y_target'].reset_index(drop=True)
    y_val = val_part['y_target'].to_numpy(dtype=float)
    w_val = val_part['weight'].to_numpy(dtype=float)
    steps = len(val_part)
    train_scale = float(np.mean(np.abs(np.diff(y_train)))) if len(y_train) > 1 else np.nan

    for model_name, spec in MODEL_SPECS.items():
        result = {
            'series_label': row['series_label'],
            'reason': row['reason'],
            'code': row['code'],
            'sub_code': row['sub_code'],
            'sub_category': row['sub_category'],
            'horizon': int(row['horizon']),
            'model': model_name,
            'train_len': len(y_train),
            'val_len': steps,
            'status': 'ok',
            'skip_reason': None,
            'skill_score': np.nan,
            'rmse': np.nan,
            'mae': np.nan,
            'mase': np.nan,
        }

        if len(y_train) < spec['min_train']:
            result['status'] = 'skipped'
            result['skip_reason'] = f"train_len<{spec['min_train']}"
            results_rows.append(result)
            continue

        try:
            pred = np.asarray(spec['func'](y_train, steps), dtype=float)
            if len(pred) != steps:
                raise ValueError('forecast_length_mismatch')
            if not np.all(np.isfinite(pred)):
                raise ValueError('non_finite_forecast')

            result['skill_score'] = weighted_skill(y_val, pred, w_val)
            result['rmse'] = weighted_rmse(y_val, pred, w_val)
            result['mae'] = weighted_mae(y_val, pred, w_val)
            result['mase'] = mase(y_val, pred, train_scale)
            predictions[(row['series_label'], model_name)] = pred
        except Exception as e:
            result['status'] = 'failed'
            result['skip_reason'] = type(e).__name__

        results_rows.append(result)

results_df = pd.DataFrame(results_rows)
ok_results = results_df[results_df['status'] == 'ok'].copy()
print(results_df['status'].value_counts())
display(results_df[['series_label', 'model', 'status', 'skill_score', 'rmse', 'mae', 'mase', 'skip_reason']].round(4))

    

## 6. Comparison tables and charts

We compare models in three complementary ways:

- a **per-series leaderboard**
- an **overall summary** across the 4 series
- visual charts that make the useful differences easy to see quickly


In [ ]:
ok_results['reason'] = pd.Categorical(ok_results['reason'], categories=REASON_ORDER, ordered=True)
ok_results = ok_results.sort_values(['reason', 'skill_score', 'rmse'], ascending=[True, False, True])

per_series_leaderboard = (
    ok_results[['series_label', 'model', 'skill_score', 'rmse', 'mae', 'mase']]
    .sort_values(['series_label', 'skill_score', 'rmse'], ascending=[True, False, True])
)

overall_summary = (
    ok_results.groupby('model')
              .agg(
                  mean_skill=('skill_score', 'mean'),
                  mean_rmse=('rmse', 'mean'),
                  mean_mae=('mae', 'mean'),
                  mean_mase=('mase', 'mean'),
                  n_ok=('model', 'size'),
              )
)

overall_summary['n_attempted'] = results_df.groupby('model').size().reindex(overall_summary.index)
overall_summary['ok_rate'] = overall_summary['n_ok'] / overall_summary['n_attempted']
overall_summary = overall_summary.sort_values(['mean_skill', 'mean_rmse'], ascending=[False, True])

best_by_series = (
    ok_results.sort_values(['reason', 'skill_score', 'rmse'], ascending=[True, False, True])
              .groupby('series_label', as_index=False)
              .first()[['series_label', 'model', 'skill_score', 'rmse', 'mae', 'mase']]
)

print('Per-series best model:')
print(best_by_series.round(4).to_string(index=False))
print('\nDetailed leaderboard:')
display(per_series_leaderboard.round(4))
print('\nOverall model summary:')
display(overall_summary.round(4))


In [ ]:
skill_pivot = (
    ok_results.pivot(index='series_label', columns='model', values='skill_score')
              .reindex(chosen['series_label'])
)

status_summary = results_df.groupby(['model', 'status']).size().unstack(fill_value=0)
status_summary = status_summary.reindex(list(MODEL_SPECS), fill_value=0)
status_cols = [col for col in ['ok', 'skipped', 'failed'] if col in status_summary.columns]
status_colors = {'ok': '#55A868', 'skipped': '#C44E52', 'failed': '#8172B3'}

fig, axes = plt.subplots(1, 3, figsize=(21, 5), gridspec_kw={'width_ratios': [2.2, 1.0, 1.2]})

sns.heatmap(
    skill_pivot,
    annot=True,
    fmt='.3f',
    cmap='RdYlGn',
    center=0,
    ax=axes[0],
    cbar_kws={'label': 'skill score'},
)
axes[0].set_title('Skill score by series × model')
axes[0].set_xlabel('model')
axes[0].set_ylabel('series')

mean_skill = overall_summary['mean_skill'].sort_values(ascending=True)
axes[1].barh(mean_skill.index, mean_skill.values, color='#4C72B0')
axes[1].set_title('Mean skill across the 4 series')
axes[1].set_xlabel('mean skill score')
axes[1].grid(True, axis='x', alpha=0.3)

status_summary[status_cols].plot(
    kind='bar',
    stacked=True,
    color=[status_colors[col] for col in status_cols],
    ax=axes[2],
)
axes[2].set_title('Fit status by model')
axes[2].set_xlabel('model')
axes[2].set_ylabel('count')
axes[2].tick_params(axis='x', rotation=45)
axes[2].legend(title='status', fontsize=8)

plt.tight_layout()
plt.show()


## 7. Forecast comparison plots

The earlier version of this notebook showed many straight lines because `Zero`, `Naive`, and `Drift` were flat or linear by construction, and two representative series only have 7 training points before the cutoff. This revised section removes those trivial baselines and plots only the **top 3 valid models** from the remaining lineup.

Some flatness can still remain on the shortest-history series because `Rolling Mean`, `SES`, and sometimes `Holt` have very little information to extrapolate from.


In [ ]:
fig, axes = plt.subplots(len(chosen), 1, figsize=(14, 12), sharex=False)
axes = np.atleast_1d(axes)
colors = ['#C44E52', '#8172B3', '#64B5CD']

for ax, (_, row) in zip(axes, chosen.iterrows()):
    series_df = get_series(row)
    train_part, val_part = split_series(series_df)

    sub = (
        ok_results[ok_results['series_label'] == row['series_label']]
        .sort_values(['skill_score', 'rmse'], ascending=[False, True])
        .head(3)
    )

    ax.plot(train_part['ts_index'], train_part['y_target'], color='black', linewidth=1.0, label='train')
    ax.plot(val_part['ts_index'], val_part['y_target'], color='#DD8452', linewidth=1.4, label='validation truth')

    for color, (_, model_row) in zip(colors, sub.iterrows()):
        pred = predictions.get((row['series_label'], model_row['model']))
        if pred is None:
            continue
        ax.plot(
            val_part['ts_index'],
            pred,
            color=color,
            linewidth=1.4,
            label=f"{model_row['model']} (skill={model_row['skill_score']:.3f})",
        )

    ax.axvline(VAL_CUTOFF, color='black', linestyle='--', linewidth=1)
    ax.set_title(f"{row['series_label']} — {row['code']} / {row['sub_code']} / H{row['horizon']}")
    ax.set_ylabel('y_target')
    ax.legend(loc='best', fontsize=8)

axes[-1].set_xlabel('ts_index')
plt.tight_layout()
plt.show()
    

## 8. Residuals and fit status

The status table keeps skipped and failed fits visible, which matters here because the short-history series do not have enough observations for the ARIMA models. The residual histograms focus only on the **best valid model for each series**.


In [ ]:
display(status_summary)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for ax, (_, row) in zip(axes, best_by_series.iterrows()):
    chosen_row = chosen[chosen['series_label'] == row['series_label']].iloc[0]
    series_df = get_series(chosen_row)
    _, val_part = split_series(series_df)
    pred = predictions[(row['series_label'], row['model'])]
    residual = val_part['y_target'].to_numpy(dtype=float) - pred

    ax.hist(residual, bins=25, color='#4C72B0', alpha=0.85)
    ax.axvline(0, color='red', linestyle='--', linewidth=1)
    ax.set_title(f"{row['series_label']} — best: {row['model']}")
    ax.set_xlabel('residual')
    ax.set_ylabel('count')

plt.tight_layout()
plt.show()


## 9. Final takeaways

This notebook is intentionally small, so the conclusions are simple:

- which model wins on each representative series
- whether the short-history stable / high-weight series are mostly handled by rolling-average or smoothing baselines
- whether the volatile / long-history series benefit more from ARIMA-style models
- what this suggests before moving on to the ML notebooks


In [ ]:
final_summary = best_by_series.copy()
final_summary = final_summary.rename(columns={'model': 'best_model'})
final_summary = final_summary.merge(
    results_df[['series_label', 'train_len', 'val_len']].drop_duplicates(),
    on='series_label',
    how='left',
)
final_summary = final_summary.sort_values('series_label').reset_index(drop=True)

print('Best model for each representative series:')
print(final_summary.round(4).to_string(index=False))

print('\nShort reading:')
for _, row in final_summary.iterrows():
    print(
        f"- {row['series_label']}: {row['best_model']} "
        f"(skill={row['skill_score']:.3f}, rmse={row['rmse']:.3f}, "
        f"train_len={int(row['train_len'])}, val_len={int(row['val_len'])})"
    )
